> Projeto Desenvolve <br>
Programação Intermediária com Python <br>
Profa. Camila Laranjeira (mila@projetodesenvolve.com.br) <br>

# 3.14 - ORM

## Exercícios

#### Q1. Conhecendo os dados
Baixe o seguinte csv onde iremos trabalhar. Ele contém informações sobre salários de profissionais de dados de uma empresa hipotética entre 2009 e 2016
* https://github.com/camilalaranjeira/python-intermediario/blob/main/salaries.csv

Suas colunas, descritas na [página do Kaggle que contém o dataset](https://www.kaggle.com/datasets/krishujeniya/salary-prediction-of-data-professions?resource=download), são:
* FIRST NAME: Primeiro nome do profissional de dados (String)
* LAST NAME: Sobrenome do profissional de dados (String)
* SEX: Gênero do profissional de dados (String: 'F' para Feminino, 'M' para Masculino)
* DOJ (Date of Joining): A data em que o profissional de dados ingressou na empresa (Data no formato MM/DD/AAAA)
* CURRENT DATE: A data atual ou a data de referência dos dados (Data no formato MM/DD/AAAA)
* DESIGNATION: O cargo ou designação do profissional de dados (String: ex., Analista, Analista Sênior, Gerente)
* AGE: Idade do profissional de dados (Integer)
* SALARY: Salário anual do profissional de dados (Float)
* UNIT: Unidade de negócios ou departamento em que o profissional de dados trabalha (String: ex., TI, Finanças, Marketing)
* LEAVES USED: Número de licenças utilizadas pelo profissional de dados (Integer)
* LEAVES REMAINING: Número de licenças restantes para o profissional de dados (Integer)
* RATINGS: Avaliações de desempenho do profissional de dados (Float)
* PAST EXP: Experiência de trabalho anterior em anos antes de ingressar na empresa atual (Float)

Na célula a seguir, **carregue os dados do CSV e dê uma olhada neles antes de seguir**.

In [ ]:
### Escreva sua resposta aqui
import pandas as pd

# 1. Carregando os dados do CSV (ajuste o caminho se o arquivo estiver em outra pasta)
df = pd.read_csv('salaries.csv')

# 2. Exibindo as 5 primeiras linhas para entender o formato visual dos dados
print("--- Primeiras linhas do DataFrame ---")
display(df.head())

# 3. Exibindo informações estruturais (nomes das colunas, tipos de dados e valores nulos)
print("\n" + "="*50 + "\n")
print("--- Informações estruturais das colunas ---")
df.info()

#### Q2. Modelando os dados
Você deve **criar um ORM com SQLAlchemy capaz de comportar os dados dessa base**.

* Crie um campo de chave primária `ID`, que deve ser incrementado automaticamente
* Os campos SEX, DESIGNATION e UNIT devem ser definidos como classes `Enum` com os possíveis valores (consulte os valores únicos dessas colunas)
* Para os outros campos, consulte os tipos de dados informados na descrição acima

In [ ]:
### Escreva sua resposta aqui
from enum import Enum as PyEnum
from datetime import date
from sqlalchemy import create_engine, String, Integer, Float, Date, Enum as SqlEnum
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, sessionmaker

# 1. Criação da classe Base para o estilo declarativo do SQLAlchemy 2.0
class Base(DeclarativeBase):
    pass

# 2. Definição das classes Enum com os valores esperados para as colunas
class SexEnum(PyEnum):
    F = 'F'
    M = 'M'

class DesignationEnum(PyEnum):
    
    ANALISTA = 'Analista'
    ANALISTA_SENIOR = 'Analista Sênior'
    GERENTE = 'Gerente'
    DIRETOR = 'Diretor'

class UnitEnum(PyEnum):
    
    TI = 'TI'
    FINANCAS = 'Finanças'
    MARKETING = 'Marketing'
    RH = 'RH'

# 3. Modelagem da Tabela via ORM
class ProfissionalDados(Base):
    __tablename__ = 'profissionais_dados'
    
    # Chave primária com autoincremento automático
    id: Mapped[int] = mapped_column(Integer, primary_key=True, autoincrement=True)
    
    # Demais campos mapeados estritamente com os tipos da descrição anterior
    first_name: Mapped[str] = mapped_column(String(50), nullable=False)
    last_name: Mapped[str] = mapped_column(String(50), nullable=False)
    
    # Campos que utilizam as estruturas Enum criadas acima
    sex: Mapped[SexEnum] = mapped_column(SqlEnum(SexEnum), nullable=False)
    designation: Mapped[DesignationEnum] = mapped_column(SqlEnum(DesignationEnum), nullable=False)
    unit: Mapped[UnitEnum] = mapped_column(SqlEnum(UnitEnum), nullable=False)
    
    # Datas, idades e métricas financeiras/desempenho
    doj: Mapped[date] = mapped_column(Date, nullable=False)
    current_date: Mapped[date] = mapped_column(Date, nullable=False)
    age: Mapped[int] = mapped_column(Integer, nullable=False)
    salary: Mapped[float] = mapped_column(Float, nullable=False)
    leaves_used: Mapped[int] = mapped_column(Integer, nullable=False)
    leaves_remaining: Mapped[int] = mapped_column(Integer, nullable=False)
    ratings: Mapped[float] = mapped_column(Float, nullable=True)  # Pode conter valores nulos
    past_exp: Mapped[float] = mapped_column(Float, nullable=False)


#### Q3. Estabelecendo uma conexão

Usando o método `create_engine` do SQLAlchemy, crie uma conexão com um novo banco de dados SQLite chamado `salarios`.

In [ ]:
### Escreva sua resposta aqui
from sqlalchemy import create_engine

# 1. Cria a conexão (engine) com um novo banco de dados SQLite local chamado 'salarios.db'
engine = create_engine("sqlite:///salarios.db", echo=False)


#### Q4. Criando as tabelas
Crie as tabelas da questão Q2 no banco `salarios`.

In [ ]:
### Escreva sua resposta aqui

Base.metadata.create_all(engine)

#### Q5. Populando

Usando o método `to_sql` da biblioteca Pandas (veja [a documentação](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_sql.html)), popule o banco `salarios` com os dados do csv que você carregou na questão Q1.
* Lembre-se de definir o parâmetro `if_exists='append'` para que as tabelas não sejam dropadas e recriadas.

In [ ]:
### Escreva sua resposta aqui
import pandas as pd

# 1. Cria uma cópia para não alterar o dataframe original da Q1
df_para_banco = df.copy()

# 2. Renomea as colunas do Pandas para letras minúsculas e mapea para os nomes da tabela ORM
df_para_banco.columns = df_para_banco.columns.str.lower().str.replace(' ', '_')

# 3. Converte as colunas de data (de string para objetos datetime/date do Python)
df_para_banco['doj'] = pd.to_datetime(df_para_banco['doj']).dt.date
df_para_banco['current_date'] = pd.to_datetime(df_para_banco['current_date']).dt.date

# 4. Inserir os dados na tabela existente usando o método to_sql
df_para_banco.to_sql(
    name='profissionais_dados',  # Nome da tabela definido em __tablename__ na Q2
    con=engine,                  # O objeto engine criado na Q3
    if_exists='append',          # Parâmetro obrigatório solicitado no enunciado
    index=False                  # O Pandas não deve criar uma coluna extra para o índice
)

print("Banco de dados populado com sucesso!")

#### Q6. Consultas SQL vs ORM

Agrupe os dados por DESIGNATION e selecione o mínimo, máximo e a média dos salários (SALARY) divididos por 12. Já que o atributo SALARY é anual, dividir por 12 nos mostrará os valores mensais.

Assumindo que a variável que armazena a sua conexão se chama `engine`, você deve realizar a query acima de três formas:
* Executando a query SQL através de uma instância de conexão retornada pelo método `engine.connect()`
* Executando a query SQL com o método `read_sql_query` do Pandas (veja [a documentação](https://pandas.pydata.org/docs/reference/api/pandas.read_sql_query.html)). Você usará mesma instância `engine.connect()` como um dos parâmetros.
* Executando uma query criada com o módulo `select` do SQLAlchemy. Sua execução deve ser feita através de um objeto `Session` do módulo `orm` do SQLAlchemy (`Session(engine)`).


In [ ]:
### Execute aqui sua query SQL com SQLAlchemy
from sqlalchemy import text

# Definindo a query SQL nativa com os cálculos mensais
query_sql = """
SELECT 
    designation,
    MIN(salary) / 12.0 AS salario_min_mensal,
    MAX(salary) / 12.0 AS salario_max_mensal,
    AVG(salary) / 12.0 AS salario_medio_mensal
FROM profissionais_dados
GROUP BY designation;
"""

# Executando via instância de conexão clássica
with engine.connect() as conexao:
    resultado = conexao.execute(text(query_sql))
    
    print(f"{'Cargo':<20} | {'Mínimo':<12} | {'Máximo':<12} | {'Média':<12}")
    print("-" * 65)
    for linha in resultado:
        print(f"{linha.designation:<20} | {linha.salario_min_mensal:<12.2f} | {linha.salario_max_mensal:<12.2f} | {linha.salario_medio_mensal:<12.2f}")

In [ ]:
### Execute aqui sua query SQL com SQLAlchemy + Pandas
import pandas as pd
from sqlalchemy import text

# Reutilizando a mesma query SQL declarada anteriormente
query_sql = """
SELECT 
    designation,
    MIN(salary) / 12.0 AS salario_min_mensal,
    MAX(salary) / 12.0 AS salario_max_mensal,
    AVG(salary) / 12.0 AS salario_medio_mensal
FROM profissionais_dados
GROUP BY designation;
"""

# Executando com Pandas usando a instância do engine.connect()
with engine.connect() as conexao:
    df_resultado = pd.read_sql_query(text(query_sql), con=conexao)

# Exibindo o DataFrame resultante formatado no Jupyter
display(df_resultado)

In [ ]:
### Execute aqui sua query com SQLAlchemy ORM
from sqlalchemy import select, func
from sqlalchemy.orm import Session

# Iniciando a sessão do ORM
with Session(engine) as session:
    # Construindo a query com o construtor select utilizando a classe ProfissionalDados mapeada na Q2
    stmt = (
        select(
            ProfissionalDados.designation,
            (func.min(ProfissionalDados.salary) / 12.0).label("min_mensal"),
            (func.max(ProfissionalDados.salary) / 12.0).label("max_mensal"),
            (func.avg(ProfissionalDados.salary) / 12.0).label("media_mensal")
        )
        .group_by(ProfissionalDados.designation)
    )
    
    # Executando e processando os resultados do ORM
    resultados_orm = session.execute(stmt).all()
    
    print(f"{'Cargo (ORM)':<20} | {'Mínimo':<12} | {'Máximo':<12} | {'Média':<12}")
    print("-" * 65)
    for cargo, minimo, maximo, media in resultados_orm:
        # Como definimos Enum na Q2, extraímos o .value dele se necessário na exibição
        nome_cargo = cargo.value if hasattr(cargo, 'value') else cargo
        print(f"{nome_cargo:<20} | {minimo:<12.2f} | {maximo:<12.2f} | {media:<12.2f}")